In [95]:
import pandas as pd
import numpy as np


# **1. Load Data**

In [96]:
df = pd.read_excel("/content/track4_mid_day_meal_procurement.xlsx")
df.head()

,procurement_id,date,school_id,vendor_name,grain_type,quantity,unit,total_cost,payment_status
0,MDM008522,2026/01/14,SCH0249,Singh Brothers,WHEAT,NaN,NaN,336/-,PENDING
1,MDM010158,2025-08-10,SCH0140,kumar general store,gehun,40.7,KGS,1221,Cleared
2,MDM007882,03/11/2025,SCH-0514,kumar general store,Oil,54.2,KGS,6504,Paid
3,MDM004875,2026/03/19,SCH-0445,sharma traders pvt ltd,RICE,0.8,50kg Bags,"Rs. 1,600",Due
4,MDM011422,16.08.2025,sch_0172,Kumar Supplies,chawal,NaN,NaN,"Rs. 2,104",Paid


# **2. Basic Profiling**

In [97]:
# Display basic information about the DataFrame
print("DataFrame Info:")
df.info()

DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12360 entries, 0 to 12359
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   procurement_id  12360 non-null  object
 1   date            12360 non-null  object
 2   school_id       12360 non-null  object
 3   vendor_name     12360 non-null  object
 4   grain_type      12360 non-null  object
 5   quantity        10451 non-null  object
 6   unit            8556 non-null   object
 7   total_cost      11714 non-null  object
 8   payment_status  10641 non-null  object
dtypes: object(9)
memory usage: 869.2+ KB


In [98]:
# Display descriptive statistics for numerical columns
print("\nDescriptive Statistics:")
display(df.describe(include='all'))


Descriptive Statistics:


,procurement_id,date,school_id,vendor_name,grain_type,quantity,unit,total_cost,payment_status
count,12360,12360,12360,12360,12360,10451.0,8556,11714,10641
unique,12000,2175,3165,12,18,1984.0,12,5552,6
top,MDM011193,2025-09-21,SCH0520,Kumar & Co.,Sarson Tel,40.1,KG,1476,Due
freq,2,18,19,1085,805,18.0,944,14,1834


In [99]:
# Check for missing values
print("\nMissing Values:")
display(df.isnull().sum())


Missing Values:


,0
procurement_id,0
date,0
school_id,0
vendor_name,0
grain_type,0
quantity,1909
unit,3804
total_cost,646
payment_status,1719


In [100]:
df.describe(include="all").T

,count,unique,top,freq
procurement_id,12360,12000,MDM011193,2
date,12360,2175,2025-09-21,18
school_id,12360,3165,SCH0520,19
vendor_name,12360,12,Kumar & Co.,1085
grain_type,12360,18,Sarson Tel,805
quantity,10451.0,1984.0,40.1,18.0
unit,8556,12,KG,944
total_cost,11714,5552,1476,14
payment_status,10641,6,Due,1834


# **3. Data Cleaning**

In [101]:
mdm_clean = df.copy()
mdm_clean.shape

(12360, 9)

In [102]:
mdm_clean.columns = (
    mdm_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

In [103]:
mdm_clean.columns

Index(['procurement_id', 'date', 'school_id', 'vendor_name', 'grain_type',
       'quantity', 'unit', 'total_cost', 'payment_status'],
      dtype='object')

In [104]:
# Clean string columns

string_columns = [
    "procurement_id",
    "school_id",
    "vendor_name",
    "grain_type",
    "unit",
    "payment_status"
]

for col in string_columns:
    mdm_clean[col] = (
        mdm_clean[col]
        .astype("string")
        .str.strip()
    )

In [105]:

mdm_clean = mdm_clean.replace(
    ["NA", "N/A", "na", "n/a", "", "NULL", "null"],
    np.nan
)

In [106]:
mdm_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12360 entries, 0 to 12359
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   procurement_id  12360 non-null  string
 1   date            12360 non-null  object
 2   school_id       12360 non-null  string
 3   vendor_name     12360 non-null  string
 4   grain_type      12360 non-null  string
 5   quantity        10451 non-null  object
 6   unit            8556 non-null   string
 7   total_cost      11714 non-null  object
 8   payment_status  10641 non-null  string
dtypes: object(3), string(6)
memory usage: 869.2+ KB


In [107]:
# Clean school_id

def clean_school_id(value):

    if pd.isna(value):
        return pd.NA

    value = str(value).strip().upper()

    # Keep only digits
    digits = ''.join(ch for ch in value if ch.isdigit())

    if digits == "":
        return pd.NA

    return f"SCH{int(digits):04d}"

In [108]:
mdm_clean["school_id"] = (
    mdm_clean["school_id"]
    .apply(clean_school_id)
    .astype("string")
)

In [109]:
mdm_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12360 entries, 0 to 12359
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   procurement_id  12360 non-null  string
 1   date            12360 non-null  object
 2   school_id       12360 non-null  string
 3   vendor_name     12360 non-null  string
 4   grain_type      12360 non-null  string
 5   quantity        10451 non-null  object
 6   unit            8556 non-null   string
 7   total_cost      11714 non-null  object
 8   payment_status  10641 non-null  string
dtypes: object(3), string(6)
memory usage: 869.2+ KB


In [110]:
mdm_clean["date"].isna().sum()

np.int64(0)

#### 2.3 Re-cleaning and refining the 'date' column

In [111]:
# Restore original 'date' column from df to mdm_clean to retry parsing correctly
mdm_clean['date'] = df['date']

def parse_date_robustly(date_str):
    if pd.isna(date_str):
        return pd.NaT
    date_str = str(date_str).strip()
    formats_to_try = [
        '%Y/%m/%d',  # e.g., 2026/01/14
        '%Y-%m-%d',  # e.g., 2025-08-10
        '%d/%m/%Y',  # e.g., 03/11/2025
        '%d.%m.%Y'   # e.g., 16.08.2025
    ]
    for fmt in formats_to_try:
        try:
            return pd.to_datetime(date_str, format=fmt)
        except ValueError:
            continue
    # If specific formats fail, try general parsing as a last resort
    try:
        return pd.to_datetime(date_str, errors='coerce')
    except ValueError:
        return pd.NaT

mdm_clean['date'] = mdm_clean['date'].apply(parse_date_robustly)
mdm_clean['date'] = mdm_clean['date'].dt.normalize() # Normalize to remove time components

print("\nMissing values in 'date' after robust parsing:")
print(mdm_clean['date'].isna().sum())
print("\nFirst 5 rows with cleaned dates:")
display(mdm_clean.head())


Missing values in 'date' after robust parsing:
0

First 5 rows with cleaned dates:


,procurement_id,date,school_id,vendor_name,grain_type,quantity,unit,total_cost,payment_status
0,MDM008522,2026-01-14,SCH0249,Singh Brothers,WHEAT,NaN,<NA>,336/-,PENDING
1,MDM010158,2025-08-10,SCH0140,kumar general store,gehun,40.7,KGS,1221,Cleared
2,MDM007882,2025-11-03,SCH0514,kumar general store,Oil,54.2,KGS,6504,Paid
3,MDM004875,2026-03-19,SCH0445,sharma traders pvt ltd,RICE,0.8,50kg Bags,"Rs. 1,600",Due
4,MDM011422,2025-08-16,SCH0172,Kumar Supplies,chawal,NaN,<NA>,"Rs. 2,104",Paid


In [112]:
mdm_clean.info(), mdm_clean.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12360 entries, 0 to 12359
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   procurement_id  12360 non-null  string        
 1   date            12360 non-null  datetime64[ns]
 2   school_id       12360 non-null  string        
 3   vendor_name     12360 non-null  string        
 4   grain_type      12360 non-null  string        
 5   quantity        10451 non-null  object        
 6   unit            8556 non-null   string        
 7   total_cost      11714 non-null  object        
 8   payment_status  10641 non-null  string        
dtypes: datetime64[ns](1), object(2), string(6)
memory usage: 869.2+ KB


(None, (12360, 9))

#### 2.4 Verify and refine `school_id`, `vendor_name`, and `grain_type`

In [113]:
# vendor_name cleaning

mdm_clean["vendor_name"].head()

,vendor_name
0,Singh Brothers
1,kumar general store
2,kumar general store
3,sharma traders pvt ltd
4,Kumar Supplies


In [114]:
mdm_clean["vendor_name"] = (
        mdm_clean["vendor_name"]
        .astype("string")
        .str.replace(
            r"\s+",
            " ",
            regex=True
        )
        .str.title()
    )

In [115]:
mdm_clean["vendor_name"].head()

,vendor_name
0,Singh Brothers
1,Kumar General Store
2,Kumar General Store
3,Sharma Traders Pvt Ltd
4,Kumar Supplies


In [116]:
mdm_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12360 entries, 0 to 12359
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   procurement_id  12360 non-null  string        
 1   date            12360 non-null  datetime64[ns]
 2   school_id       12360 non-null  string        
 3   vendor_name     12360 non-null  string        
 4   grain_type      12360 non-null  string        
 5   quantity        10451 non-null  object        
 6   unit            8556 non-null   string        
 7   total_cost      11714 non-null  object        
 8   payment_status  10641 non-null  string        
dtypes: datetime64[ns](1), object(2), string(6)
memory usage: 869.2+ KB


In [117]:
mdm_clean["quantity"].isna().sum()

np.int64(1909)

In [118]:
# grain_type cleaning

mdm_clean["grain_type"].unique()

<StringArray>
[      'WHEAT',       'gehun',         'Oil',        'RICE',      'chawal',
         'dal',        'Daal',       'Wheat',        'Atta',      'Chawal',
      'Pulses', 'Mustard Oil', 'Cooking Oil',  'Sarson Tel',         'Dal',
       'Gehun',        'Rice',     'Lentils']
Length: 18, dtype: string

In [119]:
mdm_clean["grain_type"] = (
    mdm_clean["grain_type"]
    .astype("string")
    .str.strip()
    .str.lower()
)

In [120]:
len(mdm_clean["grain_type"].unique())

13

In [121]:
grain_mapping = {
    "wheat": "Wheat",
    "gehun": "Wheat",

    "rice": "Rice",
    "chawal": "Rice",

    "dal": "Dal",
    "daal": "Dal",
    "pulses": "Dal",
    "lentils": "Dal",

    "mustard oil": "Mustard Oil",
    "sarson tel": "Mustard Oil",

    "cooking oil": "Cooking Oil",

    "atta": "Atta"
}

mdm_clean["grain_type"] = (
    mdm_clean["grain_type"]
    .replace(grain_mapping)
)

In [122]:
mdm_clean["grain_type"].unique()

<StringArray>
['Wheat', 'oil', 'Rice', 'Dal', 'Atta', 'Mustard Oil', 'Cooking Oil']
Length: 7, dtype: string

In [123]:
mdm_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12360 entries, 0 to 12359
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   procurement_id  12360 non-null  string        
 1   date            12360 non-null  datetime64[ns]
 2   school_id       12360 non-null  string        
 3   vendor_name     12360 non-null  string        
 4   grain_type      12360 non-null  string        
 5   quantity        10451 non-null  object        
 6   unit            8556 non-null   string        
 7   total_cost      11714 non-null  object        
 8   payment_status  10641 non-null  string        
dtypes: datetime64[ns](1), object(2), string(6)
memory usage: 869.2+ KB


#### 2.5 Clean 'quantity' Column

In [124]:
mdm_clean['quantity'].isna().sum()

np.int64(1909)

In [125]:
missing_quantity_pct = (
    mdm_clean["quantity"].isna().mean() * 100
)

print(f"Missing quantity: {missing_quantity_pct:.2f}%")

Missing quantity: 15.44%


In [126]:
# Convert 'quantity' to numeric, coercing errors to NaN
mdm_clean['quantity'] = pd.to_numeric(mdm_clean['quantity'], errors='coerce')

print("Missing values in 'quantity' after cleaning:")
print(mdm_clean['quantity'].isna().sum())

print("\nFirst 5 rows with cleaned quantity:")
display(mdm_clean.head())

Missing values in 'quantity' after cleaning:
3804

First 5 rows with cleaned quantity:


,procurement_id,date,school_id,vendor_name,grain_type,quantity,unit,total_cost,payment_status
0,MDM008522,2026-01-14,SCH0249,Singh Brothers,Wheat,NaN,<NA>,336/-,PENDING
1,MDM010158,2025-08-10,SCH0140,Kumar General Store,Wheat,40.7,KGS,1221,Cleared
2,MDM007882,2025-11-03,SCH0514,Kumar General Store,oil,54.2,KGS,6504,Paid
3,MDM004875,2026-03-19,SCH0445,Sharma Traders Pvt Ltd,Rice,0.8,50kg Bags,"Rs. 1,600",Due
4,MDM011422,2025-08-16,SCH0172,Kumar Supplies,Rice,NaN,<NA>,"Rs. 2,104",Paid


In [127]:
df.columns

Index(['procurement_id', 'date', 'school_id', 'vendor_name', 'grain_type',
       'quantity', 'unit', 'total_cost', 'payment_status'],
      dtype='object')

In [128]:
mdm_clean.info(), mdm_clean.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12360 entries, 0 to 12359
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   procurement_id  12360 non-null  string        
 1   date            12360 non-null  datetime64[ns]
 2   school_id       12360 non-null  string        
 3   vendor_name     12360 non-null  string        
 4   grain_type      12360 non-null  string        
 5   quantity        8556 non-null   float64       
 6   unit            8556 non-null   string        
 7   total_cost      11714 non-null  object        
 8   payment_status  10641 non-null  string        
dtypes: datetime64[ns](1), float64(1), object(1), string(6)
memory usage: 869.2+ KB


(None, (12360, 9))

#### 2.6 Clean 'unit' Column

In [129]:
df['unit'].unique()

array([nan, 'KGS', '50kg Bags', 'Grams', 'KG', 'Kgs', 'Bori', 'Sacks',
       'g', 'kg', 'Bags', 'grams', 'bags'], dtype=object)

In [130]:
mdm_clean["unit"] = (
    mdm_clean["unit"]
    .astype("string")
    .str.strip()
    .str.upper()
)

df['unit'].unique()

array([nan, 'KGS', '50kg Bags', 'Grams', 'KG', 'Kgs', 'Bori', 'Sacks',
       'g', 'kg', 'Bags', 'grams', 'bags'], dtype=object)

In [131]:
mdm_clean['unit'].head()

,unit
0,NaN
1,KGS
2,KGS
3,50kg Bags
4,NaN


In [132]:
unit_mapping = {
    "KG": "KG",
    "KGS": "KG",

    "G": "G",
    "GRAM": "G",
    "GRAMS": "G",

    "BAG": "BAG",
    "BAGS": "BAG",

    "SACK": "SACK",
    "SACKS": "SACK",

    "BORI": "BORI",

    "50KG BAG": "BAG_50KG",
    "50KG BAGS": "BAG_50KG"
}

mdm_clean["unit"] = (
    mdm_clean["unit"]
    .replace(unit_mapping)
)

In [134]:
mdm_clean['unit'].head()

,unit
0,<NA>
1,KG
2,KG
3,BAG_50KG
4,<NA>


In [135]:
mdm_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12360 entries, 0 to 12359
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   procurement_id  12360 non-null  string        
 1   date            12360 non-null  datetime64[ns]
 2   school_id       12360 non-null  string        
 3   vendor_name     12360 non-null  string        
 4   grain_type      12360 non-null  string        
 5   quantity        8556 non-null   float64       
 6   unit            8556 non-null   string        
 7   total_cost      11714 non-null  object        
 8   payment_status  10641 non-null  string        
dtypes: datetime64[ns](1), float64(1), object(1), string(6)
memory usage: 869.2+ KB


#### 2.6 Clean 'total_cost' Column

In [136]:
mdm_clean["total_cost"].head()

,total_cost
0,336/-
1,1221
2,6504
3,"Rs. 1,600"
4,"Rs. 2,104"


In [137]:
mdm_clean["total_cost"].isna().sum()

np.int64(646)

In [143]:
mdm_clean["total_cost"].dropna().sample(30, random_state=42)

,total_cost
6454,1696
5421,2151
7824,"₹1,317"
3589,1908
4288,5202
9921,3915
2037,"2,466/-"
954,₹705
6551,Rs. 924
5294,"₹6,228"


In [144]:
mdm_clean["total_cost"].value_counts().head(30)

,count
total_cost,
1476,14
972,11
1656,11
1272,11
1728,11
1368,11
936,11
2184,10
1764,10


In [146]:
mdm_clean["total_cost"] = (
    mdm_clean["total_cost"]
    .astype("string")
    .str.strip()
    .str.replace(",", "", regex=False)
)

In [147]:
mdm_clean.total_cost.head()

,total_cost
0,336/-
1,1221
2,6504
3,Rs. 1600
4,Rs. 2104


In [148]:
mdm_clean["total_cost"] = (
    mdm_clean["total_cost"]
    .str.extract(r"([-+]?\d*\.?\d+)")[0]
)

mdm_clean.total_cost.head()

,total_cost
0,336
1,1221
2,6504
3,1600
4,2104


In [149]:
mdm_clean["total_cost"] = pd.to_numeric(
    mdm_clean["total_cost"],
    errors="coerce"
)

mdm_clean.total_cost.head()

,total_cost
0,336
1,1221
2,6504
3,1600
4,2104


In [154]:
mdm_clean.head()

,procurement_id,date,school_id,vendor_name,grain_type,quantity,unit,total_cost,payment_status
0,MDM008522,2026-01-14,SCH0249,Singh Brothers,Wheat,NaN,<NA>,336,PENDING
1,MDM010158,2025-08-10,SCH0140,Kumar General Store,Wheat,40.7,KG,1221,Cleared
2,MDM007882,2025-11-03,SCH0514,Kumar General Store,oil,54.2,KG,6504,Paid
3,MDM004875,2026-03-19,SCH0445,Sharma Traders Pvt Ltd,Rice,0.8,BAG_50KG,1600,Due
4,MDM011422,2025-08-16,SCH0172,Kumar Supplies,Rice,NaN,<NA>,2104,Paid


In [155]:
mdm_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12360 entries, 0 to 12359
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   procurement_id  12360 non-null  string        
 1   date            12360 non-null  datetime64[ns]
 2   school_id       12360 non-null  string        
 3   vendor_name     12360 non-null  string        
 4   grain_type      12360 non-null  string        
 5   quantity        8556 non-null   float64       
 6   unit            8556 non-null   string        
 7   total_cost      11714 non-null  Int64         
 8   payment_status  10641 non-null  string        
dtypes: Int64(1), datetime64[ns](1), float64(1), string(6)
memory usage: 881.3 KB


#### 2.6 Clean 'payment_status' Column

In [156]:
df.payment_status.head()

,payment_status
0,PENDING
1,Cleared
2,Paid
3,Due
4,Paid


In [157]:
df.payment_status.isna().sum()

np.int64(1719)

In [158]:
df.payment_status.unique()

array(['PENDING', 'Cleared', 'Paid', 'Due', nan, 'paid', 'Pending'],
      dtype=object)

In [159]:
# Standardize payment_status

mdm_clean["payment_status"] = (
    mdm_clean["payment_status"]
    .astype("string")
    .str.strip()
    .str.lower()
)

In [160]:
payment_mapping = {
    "pending": "Pending",
    "cleared": "Cleared",
    "paid": "Paid",
    "due": "Due"
}

mdm_clean["payment_status"] = (
    mdm_clean["payment_status"]
    .replace(payment_mapping)
)

In [161]:
print(mdm_clean["payment_status"].unique())

<StringArray>
['Pending', 'Cleared', 'Paid', 'Due', <NA>]
Length: 5, dtype: string


In [162]:
mdm_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12360 entries, 0 to 12359
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   procurement_id  12360 non-null  string        
 1   date            12360 non-null  datetime64[ns]
 2   school_id       12360 non-null  string        
 3   vendor_name     12360 non-null  string        
 4   grain_type      12360 non-null  string        
 5   quantity        8556 non-null   float64       
 6   unit            8556 non-null   string        
 7   total_cost      11714 non-null  Int64         
 8   payment_status  10641 non-null  string        
dtypes: Int64(1), datetime64[ns](1), float64(1), string(6)
memory usage: 881.3 KB


# **4.Basic validation of MDM**

In [163]:
print("Rows:", len(mdm_clean))
print("Columns:", len(mdm_clean.columns))

print("\nMissing values:")
print(mdm_clean.isna().sum())

print("\nDuplicate rows:")
print(mdm_clean.duplicated().sum())

Rows: 12360
Columns: 9

Missing values:
procurement_id       0
date                 0
school_id            0
vendor_name          0
grain_type           0
quantity          3804
unit              3804
total_cost         646
payment_status    1719
dtype: int64

Duplicate rows:
360


In [164]:
print("Duplicate procurement IDs:",
      mdm_clean["procurement_id"].duplicated().sum())

Duplicate procurement IDs: 360


In [165]:
duplicate_ids = mdm_clean[
    mdm_clean["procurement_id"].duplicated(keep=False)
].sort_values("procurement_id")

duplicate_ids.head(20)

,procurement_id,date,school_id,vendor_name,grain_type,quantity,unit,total_cost,payment_status
477,MDM000020,2026-03-28,SCH0320,Kumar General Store,Dal,0.766,BAG,3446,Pending
2681,MDM000020,2026-03-28,SCH0320,Kumar General Store,Dal,0.766,BAG,3446,Pending
386,MDM000071,2026-03-01,SCH0410,S. Agro Works,Rice,35.000,KG,1400,Cleared
2951,MDM000071,2026-03-01,SCH0410,S. Agro Works,Rice,35.000,KG,1400,Cleared
558,MDM000096,2026-01-27,SCH0416,Sharma Traders,Rice,NaN,<NA>,764,Pending
7205,MDM000096,2026-01-27,SCH0416,Sharma Traders,Rice,NaN,<NA>,764,Pending
238,MDM000100,2025-08-07,SCH0409,Goyal Rice Mill,Mustard Oil,NaN,<NA>,5700,<NA>
12333,MDM000100,2025-08-07,SCH0409,Goyal Rice Mill,Mustard Oil,NaN,<NA>,5700,<NA>
439,MDM000102,2025-04-04,SCH0316,Kumar & Co.,Dal,40.500,KG,3645,Cleared
1326,MDM000102,2025-04-04,SCH0316,Kumar & Co.,Dal,40.500,KG,3645,Cleared


In [166]:
duplicate_rows = mdm_clean[
    mdm_clean.duplicated(keep=False)
]

print("Rows involved in duplicate groups:",
      len(duplicate_rows))

Rows involved in duplicate groups: 720


In [168]:
# Check whether duplicate procurement IDs are also exact duplicate rows

duplicate_records = mdm_clean[
    mdm_clean["procurement_id"].duplicated(keep=False)
]

print(
    "Duplicate ID rows:",
    len(duplicate_records)
)

print(
    "Exact duplicate rows:",
    duplicate_records.duplicated().sum()
)

Duplicate ID rows: 720
Exact duplicate rows: 360


In [169]:
mdm_clean = mdm_clean.drop_duplicates()

In [170]:
mdm_clean = mdm_clean.reset_index(drop=True)

In [171]:
print("Rows after removing duplicates:", len(mdm_clean))

print(
    "Duplicate rows:",
    mdm_clean.duplicated().sum()
)

print(
    "Duplicate procurement IDs:",
    mdm_clean["procurement_id"].duplicated().sum()
)

Rows after removing duplicates: 12000
Duplicate rows: 0
Duplicate procurement IDs: 0


Duplicate Record Handling

The MDM procurement dataset initially contained 12,360 records. We identified 360 exact duplicate records, corresponding to 360 duplicated procurement IDs. Inspection confirmed that the duplicated records contained identical transaction information across all columns. These 360 exact duplicate records were removed to prevent double-counting in expenditure, quantity, procurement, vendor, and payment analyses. The resulting dataset contains 12,000 unique procurement transactions.

In [172]:
missing_summary = pd.DataFrame({
    "missing_count": mdm_clean.isna().sum(),
    "missing_percentage": (
        mdm_clean.isna().mean() * 100
    ).round(2)
})

missing_summary

,missing_count,missing_percentage
procurement_id,0,0.00
date,0,0.00
school_id,0,0.00
vendor_name,0,0.00
grain_type,0,0.00
quantity,3693,30.78
unit,3693,30.78
total_cost,625,5.21
payment_status,1668,13.90


In [173]:
print(
    "Duplicate procurement IDs:",
    mdm_clean["procurement_id"].duplicated().sum()
)

Duplicate procurement IDs: 0


In [174]:
negative_quantity = mdm_clean[
    mdm_clean["quantity"] < 0
]

print("Negative quantities:", len(negative_quantity))

Negative quantities: 0


In [175]:
mdm_clean["quantity"].describe()

,quantity
count,8307.000000
mean,9868.390060
std,17496.690071
min,0.202000
25%,1.045000
50%,35.000000
75%,15900.000000
max,60000.000000


In [176]:
negative_cost = mdm_clean[
    mdm_clean["total_cost"] < 0
]

print("Negative costs:", len(negative_cost))

Negative costs: 0


In [177]:
quantity_unit_check = pd.DataFrame({
    "quantity_missing": mdm_clean["quantity"].isna(),
    "unit_missing": mdm_clean["unit"].isna()
})

pd.crosstab(
    quantity_unit_check["quantity_missing"],
    quantity_unit_check["unit_missing"]
)

unit_missing,False,True
quantity_missing,,
False,8307,0
True,0,3693


In [178]:
quantity_without_unit = mdm_clean[
    mdm_clean["quantity"].notna() &
    mdm_clean["unit"].isna()
]

print(
    "Quantity present but unit missing:",
    len(quantity_without_unit)
)

Quantity present but unit missing: 0


# **5. Install cleaned data**

In [179]:
import os

os.makedirs("/content/data/cleaned", exist_ok=True)

In [180]:
output_path = "/content/data/cleaned/cleaned_mid_day_meal_procurement.csv"

mdm_clean.to_csv(
    output_path,
    index=False
)

print("Saved:", output_path)

Saved: /content/data/cleaned/cleaned_mid_day_meal_procurement.csv
